In [ ]:
# Install required package(s) in the notebook kernel
# Note: using %pip ensures installation into the notebook's Python environment.
# %pip install --quiet tensorflow opencv-python

import cv2
import numpy as np
import tensorflow as tf
from IPython.display import display, clear_output
from PIL import Image

In [ ]:
# ------------------------------
# 1. Custom MLP used by your CCT
# ------------------------------
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = tf.keras.layers.Dense(units, activation=tf.nn.gelu)(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)
    return x

# ------------------------------
# 2. Load your trained CCT model
# ------------------------------
model = tf.keras.models.load_model(
    "path_to_file/my_cct_model.keras",
    custom_objects={"mlp": mlp}
)

emotion_labels = [
    "Angry", "Disgust", "Fear",
    "Happy", "Sad", "Surprise", "Neutral"
]

# ------------------------------
# 3. Start webcam
# ------------------------------
cap = cv2.VideoCapture(0)
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

    if len(faces) > 0:

        # === Select largest face ===
        faces = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
        (x, y, w, h) = faces[0]

        # Add face padding to mimic FER2013
        pad = int(0.15 * w)
        x1 = max(x - pad, 0)
        y1 = max(y - pad, 0)
        x2 = min(x + w + pad, gray.shape[1])
        y2 = min(y + h + pad, gray.shape[0])

        face_gray = gray[y1:y2, x1:x2]

        # === Sharpen ===
        kernel = np.array([[0, -1, 0],
                        [-1, 5,-1],
                        [0, -1, 0]])
        face_gray = cv2.filter2D(face_gray, -1, kernel)

        # === CLAHE (FER2013-like) ===
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        face_gray = clahe.apply(face_gray)

        # === Resize and normalize ===
        roi = cv2.resize(face_gray, (48, 48))
        roi = roi.astype("float32") / 255.0
        roi = np.expand_dims(roi, axis=-1)
        roi = np.expand_dims(roi, axis=0)

        # Prediction
        pred = model.predict(roi, verbose=0)
        label = emotion_labels[np.argmax(pred)]

        # Draw UI
        cv2.putText(frame, label, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255,255,255), 2)

    cv2.imshow("Emotion Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


2025-12-06 08:33:29.140 Python[3276:102706] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.
2025-12-06 08:33:30.336107: I external/local_xla/xla/service/service.cc:163] XLA service 0x31c705d60 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2025-12-06 08:33:30.336119: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
2025-12-06 08:33:30.356588: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1765028010.729251  103634 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


: 